# Exploratory Data Analysis — Device Storage Growth Forecaster

This notebook explores the synthetic storage-usage dataset, checks data quality, studies user-profile behavior, and prepares ideas for forecasting models.

## Goals
- Understand dataset shape and schema
- Compare storage behavior across profiles
- Inspect cleanup events and daily changes
- Identify useful forecasting features
- Build intuition for why the baseline model underfits

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
DATA_PATH = Path('../data/synthetic/synthetic_storage_usage.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df.head()

In [ ]:
print('Shape:', df.shape)
print('Users:', df['user_id'].nunique())
print('Profiles:', df['profile'].unique())
print('Date range:', df['date'].min(), 'to', df['date'].max())
df.info()

## Missing values and descriptive statistics

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])
df.describe(numeric_only=True).T

## Profile-level summary

In [ ]:
profile_summary = (df.groupby('profile')
    .agg(users=('user_id', 'nunique'),
         avg_capacity_gb=('total_capacity_gb', 'mean'),
         avg_used_gb=('used_gb', 'mean'),
         avg_used_pct=('used_pct', 'mean'),
         avg_daily_delta_gb=('daily_delta_gb', 'mean'),
         cleanup_rate=('cleanup_event', 'mean'))
    .round(3))
profile_summary

## Average growth curves by profile

In [ ]:
growth = df.groupby(['profile', 'day_index'])['used_gb'].mean().reset_index()
plt.figure(figsize=(11,5))
for profile in growth['profile'].unique():
    sub = growth[growth['profile'] == profile]
    plt.plot(sub['day_index'], sub['used_gb'], label=profile, linewidth=2)
plt.title('Average Used Storage Over Time by Profile')
plt.xlabel('Day index')
plt.ylabel('Used GB')
plt.legend()
plt.show()

## Daily change distribution

In [ ]:
subset = df[(df['daily_delta_gb'] > -50) & (df['daily_delta_gb'] < 80)].copy()
plt.figure(figsize=(10,5))
sns.boxplot(data=subset, x='profile', y='daily_delta_gb')
plt.xticks(rotation=15)
plt.title('Daily Storage Change by Profile')
plt.show()

## Cleanup behavior over time

In [ ]:
monthly_cleanup = (df.assign(month=df['date'].dt.to_period('M').astype(str))
    .groupby(['month', 'profile'])['cleanup_event']
    .mean()
    .reset_index())
plt.figure(figsize=(12,5))
sns.lineplot(data=monthly_cleanup, x='month', y='cleanup_event', hue='profile')
plt.xticks(rotation=45, ha='right')
plt.title('Monthly Cleanup Event Rate by Profile')
plt.show()

## Correlation heatmap

In [ ]:
cols = ['used_gb','free_gb','used_pct','photos_gb','videos_gb','apps_gb','documents_gb','system_gb','other_gb','daily_delta_gb','cleanup_event']
corr = df[cols].corr(numeric_only=True)
plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

## Forecasting takeaways
- A single global linear model is too simple for these different profile behaviors.
- Lag features like yesterday, 7-day, and 30-day used storage should be helpful.
- Cleanup events create abrupt drops, which simple trend models miss.
- Profile-aware models or profile-specific models are likely to perform better.
- Prophet and XGBoost are strong next candidates for comparison.